In [20]:
import argparse
import logging
import os
from functools import partial
from multiprocessing import Pool
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import polars as pl
from scipy.signal import stft
from scipy.stats import kurtosis, skew
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [4]:
path = "../silver_data/silver_layer.parquet"

In [5]:
Path(path).exists()

True

In [6]:
df = pl.read_parquet(path)

In [7]:
df.head()

trial,SpindleAccX_mean,SpindleAccX_std,SpindleAccX_rms,SpindleAccX_kurtosis,SpindleAccX_skewness,SpindleAccX_peak_to_peak,SpindleAccX_crest_factor,SpindleAccX_shape_factor,SpindleAccX_impulse_factor,SpindleAccX_margin_factor,SpindleAccX_energy,SpindleAccX_spectral_kurtosis_mean,SpindleAccX_spectral_kurtosis_std,SpindleAccX_spectral_kurtosis_skewness,SpindleAccX_spectral_kurtosis_kurtosis,SpindleAccY_mean,SpindleAccY_std,SpindleAccY_rms,SpindleAccY_kurtosis,SpindleAccY_skewness,SpindleAccY_peak_to_peak,SpindleAccY_crest_factor,SpindleAccY_shape_factor,SpindleAccY_impulse_factor,SpindleAccY_margin_factor,SpindleAccY_energy,SpindleAccY_spectral_kurtosis_mean,SpindleAccY_spectral_kurtosis_std,SpindleAccY_spectral_kurtosis_skewness,SpindleAccY_spectral_kurtosis_kurtosis,SpindleAccZ_mean,SpindleAccZ_std,SpindleAccZ_rms,SpindleAccZ_kurtosis,SpindleAccZ_skewness,SpindleAccZ_peak_to_peak,…,PlateLFAccZ_impulse_factor,PlateLFAccZ_margin_factor,PlateLFAccZ_energy,PlateLFAccZ_spectral_kurtosis_mean,PlateLFAccZ_spectral_kurtosis_std,PlateLFAccZ_spectral_kurtosis_skewness,PlateLFAccZ_spectral_kurtosis_kurtosis,Power_mean,Power_std,Power_rms,Power_kurtosis,Power_skewness,Power_peak_to_peak,Power_crest_factor,Power_shape_factor,Power_impulse_factor,Power_margin_factor,Power_energy,Power_spectral_kurtosis_mean,Power_spectral_kurtosis_std,Power_spectral_kurtosis_skewness,Power_spectral_kurtosis_kurtosis,PlateHFAccZ_mean,PlateHFAccZ_std,PlateHFAccZ_rms,PlateHFAccZ_kurtosis,PlateHFAccZ_skewness,PlateHFAccZ_peak_to_peak,PlateHFAccZ_crest_factor,PlateHFAccZ_shape_factor,PlateHFAccZ_impulse_factor,PlateHFAccZ_margin_factor,PlateHFAccZ_energy,PlateHFAccZ_spectral_kurtosis_mean,PlateHFAccZ_spectral_kurtosis_std,PlateHFAccZ_spectral_kurtosis_skewness,PlateHFAccZ_spectral_kurtosis_kurtosis
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Segmented_Machining_ToolWear""",0.023172,0.242879,0.243982,-0.304173,-0.150652,2.483676,5.998769,1.237785,7.425184,8.741595,1.9511e6,0.080423,0.353309,2.064734,25.290528,0.013388,0.341197,0.34146,-1.1337,0.14561,2.385839,3.639937,1.15868,4.217523,4.736784,3.8216e6,0.19783,0.337873,1.330806,9.766452,0.031622,0.14071,0.144219,-0.417062,0.016271,1.602558,…,5.686405,6.450838,2.6797e6,0.237757,0.621371,6.412159,99.176186,0.68999,0.115256,0.69955,12.229566,-3.163261,0.787987,1.256907,1.013855,1.274322,1.288647,1.6040e7,-0.100611,0.585421,2.393754,23.614221,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Segmented_Spindle5000_Unbalanc…",-0.027732,0.691068,0.691625,0.229702,-0.130671,6.230416,4.717142,1.296059,6.113695,7.472437,2.1526e7,0.115013,0.41725,6.476397,183.72439,-0.008454,0.414825,0.414911,0.304185,-0.028103,4.369045,5.550856,1.301411,7.223946,8.851702,7.7469e6,0.120852,0.417472,3.255665,32.327435,0.037015,0.649336,0.65039,0.166375,-0.06475,6.787333,…,2.74899,2.815035,6940.575467,0.0286,0.401248,7.515117,200.243923,0.221109,1.995563,2.007775,2.186427,0.824852,11.537525,3.512934,1.778161,6.246562,10.753347,1.8140e8,0.04181,0.718218,13.191186,289.794581,-0.154439,0.016921,0.155363,0.000633,0.000578,0.18804,1.627022,1.005984,1.636758,1.641739,1.0862e6,-0.006339,0.361391,17.367141,800.495149
"""Segmented_Linear_Heavy""",-0.024697,0.253372,0.254573,-0.08596,0.312432,1.481005,2.963447,1.243672,3.685555,4.283569,3.6424e6,-0.002087,0.372965,9.202823,233.442751,-0.010487,0.029074,0.030908,0.122928,0.140921,0.453847,8.3767,1.23566,10.350753,12.073716,53690.628262,-0.000288,0.359164,7.377321,163.738918,0.035918,0.06209,0.071731,-0.116531,0.175954,0.575685,…,3.551083,3.697031,7970.716688,0.000728,0.353125,7.990545,255.54745,-0.002139,0.004655,0.005123,-0.581912,-0.002273,0.073076,7.529879,1.203907,9.065273,10.404131,1475.258059,0.216755,0.754901,6.166181

In [16]:
feature_cols = df.columns[1:-1]

In [17]:
feature_cols

['SpindleAccX_mean',
 'SpindleAccX_std',
 'SpindleAccX_rms',
 'SpindleAccX_kurtosis',
 'SpindleAccX_skewness',
 'SpindleAccX_peak_to_peak',
 'SpindleAccX_crest_factor',
 'SpindleAccX_shape_factor',
 'SpindleAccX_impulse_factor',
 'SpindleAccX_margin_factor',
 'SpindleAccX_energy',
 'SpindleAccX_spectral_kurtosis_mean',
 'SpindleAccX_spectral_kurtosis_std',
 'SpindleAccX_spectral_kurtosis_skewness',
 'SpindleAccX_spectral_kurtosis_kurtosis',
 'SpindleAccY_mean',
 'SpindleAccY_std',
 'SpindleAccY_rms',
 'SpindleAccY_kurtosis',
 'SpindleAccY_skewness',
 'SpindleAccY_peak_to_peak',
 'SpindleAccY_crest_factor',
 'SpindleAccY_shape_factor',
 'SpindleAccY_impulse_factor',
 'SpindleAccY_margin_factor',
 'SpindleAccY_energy',
 'SpindleAccY_spectral_kurtosis_mean',
 'SpindleAccY_spectral_kurtosis_std',
 'SpindleAccY_spectral_kurtosis_skewness',
 'SpindleAccY_spectral_kurtosis_kurtosis',
 'SpindleAccZ_mean',
 'SpindleAccZ_std',
 'SpindleAccZ_rms',
 'SpindleAccZ_kurtosis',
 'SpindleAccZ_skewness',

In [18]:
X = df.select(feature_cols).to_numpy()

In [19]:
X

array([[ 2.31720037e-02,  2.42879039e-01,  2.43981903e-01, ...,
                    nan,             nan,             nan],
       [-2.77322027e-02,  6.91068426e-01,  6.91624642e-01, ...,
        -6.33947152e-03,  3.61391017e-01,  1.73671408e+01],
       [-2.46966884e-02,  2.53372389e-01,  2.54573160e-01, ...,
        -1.53319476e-02,  3.39394314e-01,  1.64675242e+01],
       ...,
       [ 2.78839827e-02,  2.25546600e-01,  2.27263691e-01, ...,
                    nan,             nan,             nan],
       [ 2.42978363e-02,  2.11899938e-01,  2.13288463e-01, ...,
                    nan,             nan,             nan],
       [-2.66922360e-02,  1.18629720e+00,  1.18659746e+00, ...,
        -1.27084791e-02,  3.66475860e-01,  1.51171641e+01]],
      shape=(15, 119))

In [21]:
X_scaled = StandardScaler().fit_transform(X)

In [22]:
pca = PCA(n_components=3)
components = pca.fit_transform(X_scaled)

ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values